In [ ]:
# =====================================================================
# Pareto Front — NSGA-II Multi-Objective Optimization
# Objectives:  f1 = area (minimize),  f2 = grid outage (minimize)
# Channel model, grid, weights — all unchanged from grid_search.ipynb
# =====================================================================

import torch
import numpy as np
import math
import matplotlib.pyplot as plt
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ==========================================
# 1. System Parameters (unchanged)
# ==========================================
room_x, room_y, room_z_max = 10.0, 10.0, 3.0
x_BS, y_BS, z_BS = 10.0, -100.0, -10.0
E = 8; d_B = 0.075; lambda_val = 0.075; L1 = 2
d_ref = abs(y_BS) * 1.5; P_BS_dBm = 40.0; R_th = 0.2
N0_dBm_Hz = -174.0; B = 20e6; p_m = 1.0 / 5.0; n_users = 5
P_BS = 10**(P_BS_dBm / 10.0) * 1e-3
N0 = 10**(N0_dBm_Hz / 10.0) * 1e-3 * B

# Variable bounds (aligned with MATLAB lb/ub)
x_min, x_max = 0.2, 9.8
z_min, z_max = 0.2, 2.8
L_min, L_max = 0.1, 9.8
W_min, W_max = 0.1, 2.8   # Lz → W for clarity in Pareto context

# ==========================================
# 2. Spatial Grid (unchanged)
# ==========================================
GRID_RES = 200
Z_FIXED = 1.5
x_vals = torch.linspace(0, room_x, GRID_RES, device=device)
y_vals = torch.linspace(0, room_y, GRID_RES, device=device)
Xg, Yg = torch.meshgrid(x_vals, y_vals, indexing='ij')
grid_locs = torch.stack([Xg.flatten(), Yg.flatten(),
                         torch.full_like(Xg.flatten(), Z_FIXED)], dim=1)
N_GRID = grid_locs.shape[0]
print(f'Grid: {GRID_RES}x{GRID_RES} = {N_GRID} points')

# ==========================================
# 3. Hotspot Weights (unchanged)
# ==========================================
hotspot_center = torch.tensor([room_x/2, room_y/2], device=device)
sigma_h = 2.5
dist_sq = (grid_locs[:,0]-hotspot_center[0])**2 + (grid_locs[:,1]-hotspot_center[1])**2
f_s = 1.0/(2.0*math.pi*sigma_h**2) * torch.exp(-dist_sq/(2.0*sigma_h**2))
f_u = torch.full_like(f_s, 1.0/(room_x*room_y))
grid_weights = (0.7*f_s + 0.3*f_u)
grid_weights = grid_weights / grid_weights.sum()
print(f'Weights ready, hotspot/edge = {grid_weights.max().item()/grid_weights.min().item():.1f}x')

# ==========================================
# 4. NLoS Params (fixed seed, one-time)
# ==========================================
np.random.seed(42)
_nlos_psi = torch.tensor(2*np.pi*np.random.rand(N_GRID), dtype=torch.float32, device=device)
_nlos_tt  = torch.tensor(-np.pi+2*np.pi*np.random.rand(N_GRID), dtype=torch.float32, device=device)
_nlos_eta = torch.tensor(10**((-15+5*np.random.rand(N_GRID))/10), dtype=torch.float32, device=device)

# ==========================================
# 5. Channel Model (unchanged from search_channel.m)
# ==========================================
def equivalent_farfield_channel_pytorch(window_params, locs):
    if not isinstance(window_params, torch.Tensor):
        window_params = torch.tensor(window_params, dtype=torch.float32, device=device)
    xc, zc, Lx, Lz = window_params[0], window_params[1], window_params[2], window_params[3]
    xu, yu, zu = locs[:,0], locs[:,1], locs[:,2]

    dx_BS=xc-x_BS; dy_BS=torch.tensor(0.0-y_BS,device=device); dz_BS=zc-z_BS
    R_BW=torch.sqrt(dx_BS**2+dy_BS**2+dz_BS**2)
    theta_BW=torch.atan2(dy_BS,dx_BS); phi_BW=torch.acos(dz_BS/R_BW)
    k_tx=torch.sin(phi_BW)*torch.cos(theta_BW); k_tz=torch.cos(phi_BW)

    x1,z1=xc-Lx/2,zc-Lz/2; x2,z2=xc-Lx/2,zc+Lz/2
    x3,z3=xc+Lx/2,zc-Lz/2; x4,z4=xc+Lx/2,zc+Lz/2

    def ray_dir(xs,zs):
        dx=xs-x_BS; dy=torch.tensor(0.0-y_BS,device=device); dz=zs-z_BS
        L=torch.sqrt(dx**2+dy**2+dz**2); return dx/L,dy/L,dz/L
    ux1,_,uz1=ray_dir(x1,z1); ux2,_,uz2=ray_dir(x2,z2)
    ux3,_,uz3=ray_dir(x3,z3); ux4,_,uz4=ray_dir(x4,z4)

    dx_WU=xu-x_BS; dy_WU=yu-y_BS; dz_WU=zu-z_BS
    L_USER=torch.sqrt(dx_WU**2+dy_WU**2+dz_WU**2)
    ux_user=dx_WU/L_USER; uz_user=dz_WU/L_USER

    ux_min=torch.min(torch.stack([ux1,ux2,ux3,ux4]))
    ux_max=torch.max(torch.stack([ux1,ux2,ux3,ux4]))
    uz_min=torch.min(torch.stack([uz1,uz2,uz3,uz4]))
    uz_max=torch.max(torch.stack([uz1,uz2,uz3,uz4]))

    beta=1000.0
    inx=torch.sigmoid(beta*(ux_user-ux_min))*torch.sigmoid(beta*(ux_max-ux_user))
    inz=torch.sigmoid(beta*(uz_user-uz_min))*torch.sigmoid(beta*(uz_max-uz_user))
    illum=inx*inz

    dx_WU2=xu-xc; dy_WU2=yu; dz_WU2=zu-zc
    R_WU=torch.sqrt(dx_WU2**2+dy_WU2**2+dz_WU2**2)
    t1=dx_WU2/R_WU; t2=dz_WU2/R_WU
    ax=(1.0-illum)*(k_tx-t1); az=(1.0-illum)*(k_tz-t2)
    sincx=torch.sinc((math.pi/lambda_val)*Lx*ax)
    sincz=torch.sinc((math.pi/lambda_val)*Lz*az)

    n_ant=torch.arange(E,dtype=torch.float32,device=device)

    # Path 1 (LoS)
    ph1=(2.0*math.pi/lambda_val)*d_B*n_ant*torch.sin(theta_BW)
    a1=(1.0/math.sqrt(E))*torch.complex(torch.cos(ph1),torch.sin(ph1))
    v1m=torch.tensor(lambda_val/(4.0*math.pi),device=device)/R_BW
    v1p=torch.tensor(-(2.0*math.pi/lambda_val),device=device)*R_BW
    v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p))
    H1=v1*a1.conj()

    # Path 2 (NLoS)
    ph2=(2.0*math.pi/lambda_val)*d_B*n_ant.unsqueeze(0)*torch.sin(_nlos_tt).unsqueeze(1)
    a2=(1.0/math.sqrt(E))*torch.complex(torch.cos(ph2),torch.sin(ph2))
    v2m=_nlos_eta*(lambda_val/(4.0*math.pi*d_ref))
    v2=torch.complex(v2m*torch.cos(_nlos_psi),v2m*torch.sin(_nlos_psi))
    H2=v2.unsqueeze(1)*a2.conj()

    H_total=math.sqrt(E/L1)*(H1.unsqueeze(0)+H2)
    fm=(Lx*Lz*sincx*sincz)/(lambda_val*R_WU)
    fp=torch.tensor(-(2.0*math.pi/lambda_val),device=device)*(k_tx*xc+k_tz*zc)-(math.pi/2.0)
    factor=torch.complex(fm*torch.cos(fp),fm*torch.sin(fp))
    return H_total*factor.unsqueeze(1)

# ==========================================
# 6. Grid Outage Evaluator (deterministic, returns float)
# ==========================================
def compute_grid_outage(xc, zc, Lx, Lz):
    """Single forward pass: window params → weighted outage (Python float)"""
    theta = torch.tensor([xc, zc, Lx, Lz], dtype=torch.float32, device=device)
    H_eq = equivalent_farfield_channel_pytorch(theta, grid_locs)
    H_w = torch.sum(H_eq, dim=1)/math.sqrt(E)
    dp = (torch.abs(H_w)**2)*p_m*P_BS
    intf = (n_users-1)*dp
    sinr = dp/(intf+N0)
    rate = torch.log2(1.0+sinr)
    outage_bits = (rate < R_th).float()
    return float(torch.sum(outage_bits*grid_weights).item())

# Quick sanity check
print(f'Sanity: [5,1.5,0.3,0.3] → outage={compute_grid_outage(5.0,1.5,0.3,0.3)*100:.2f}%\n')

# ==========================================
# 7. pymoo Problem Definition
#    f1 = area  (Lx * Lz)
#    f2 = grid outage (weighted spatial integral)
# ==========================================
class EMWindowProblem(ElementwiseProblem):
    def __init__(self):
        super().__init__(
            n_var=4,
            n_obj=2,
            n_ieq_constr=4,
            xl=np.array([x_min, z_min, L_min, W_min]),
            xu=np.array([x_max, z_max, L_max, W_max])
        )

    def _evaluate(self, x, out, *args, **kwargs):
        xc, zc, Lx, Lz = x[0], x[1], x[2], x[3]

        # Inequality constraints: g_i(x) <= 0  for feasibility
        # Aligned with MATLAB nonlcon
        g1 = Lx/2 - xc               # -(xc-Lx/2) >= 0  →  Lx/2 - xc <= 0
        g2 = xc + Lx/2 - room_x       # xc+Lx/2 - room_x <= 0
        g3 = Lz/2 - zc               # -(zc-Lz/2) >= 0  →  Lz/2 - zc <= 0
        g4 = zc + Lz/2 - room_z_max   # zc+Lz/2 - room_z_max <= 0

        outage = compute_grid_outage(xc, zc, Lx, Lz)
        area = Lx * Lz

        out["F"] = [area, outage]
        out["G"] = [g1, g2, g3, g4]

# ==========================================
# 8. NSGA-II Configuration
# ==========================================
problem = EMWindowProblem()

algorithm = NSGA2(
    pop_size=300,
    n_offsprings=150,
    sampling=FloatRandomSampling(),
    crossover=SBX(prob=0.9, eta=15),
    mutation=PM(prob=0.25, eta=20),
    eliminate_duplicates=True
)

termination = get_termination("n_gen", 200)

print("="*60)
print(f" NSGA-II: pop=300, offspring=150, 200 generations")
print(f" Objectives: f1=area, f2=grid_outage")
print(f" Constraints: nonlcon boundary (4 inequalities)")
print(f" Total evaluations: ~{300*200} (all on GPU grid)")
print("="*60)

# ==========================================
# 9. Run Optimization
# ==========================================
res = minimize(problem, algorithm, termination, seed=42, verbose=True)

print(f"\nDone. Population size: {len(res.F)}")
print(f"Objective range: area [{res.F[:,0].min():.4f}, {res.F[:,0].max():.4f}]")
print(f"                 outage [{res.F[:,1].min()*100:.2f}%, {res.F[:,1].max()*100:.2f}%]")

# ==========================================
# 10. Pareto Front Visualization
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Main Pareto front
ax = axes[0]
ax.scatter(res.F[:,1]*100, res.F[:,0], c='steelblue', s=8, alpha=0.5, label='All evaluated')

ax.set_xlabel('Grid Outage [%]', fontsize=12)
ax.set_ylabel('Window Area [m²]', fontsize=12)
ax.set_title(f'Pareto Front: NSGA-II (pop=300, gen=200)', fontsize=13)
ax.axvline(x=10, color='red', linestyle='--', alpha=0.5, label='10% outage target')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)

# Annotate the knee region: smallest area with outage ≤ 10%
feasible = res.F[:,1] <= 0.10
if feasible.any():
    best_idx = np.argmin(res.F[feasible, 0])
    best_area = res.F[feasible, 0][best_idx]
    best_out = res.F[feasible, 1][best_idx]
    ax.plot(best_out*100, best_area, 'r*', markersize=14, markeredgewidth=1.5,
            label=f'Best feasible: {best_area:.4f} m² @ {best_out*100:.1f}%')
    ax.legend(fontsize=9)

# Zoomed Pareto front (clean: only Pareto-optimal points after filtering)
ax2 = axes[1]

# Simple non-dominated filter to show clean frontier
F = res.F.copy()
is_dominated = np.zeros(len(F), dtype=bool)
for i in range(len(F)):
    for j in range(len(F)):
        if i != j:
            if (F[j,0] <= F[i,0] and F[j,1] <= F[i,1]) and \
               (F[j,0] < F[i,0] or F[j,1] < F[i,1]):
                is_dominated[i] = True
                break
pareto_F = F[~is_dominated]
sort_idx = np.argsort(pareto_F[:, 1])
pareto_F = pareto_F[sort_idx]

ax2.plot(pareto_F[:,1]*100, pareto_F[:,0], 'o-', color='darkgreen',
         markersize=5, linewidth=1.5, label=f'Pareto-optimal ({len(pareto_F)} pts)')
ax2.set_xlabel('Grid Outage [%]', fontsize=12)
ax2.set_ylabel('Window Area [m²]', fontsize=12)
ax2.set_title('Pareto-Optimal Frontier', fontsize=13)
ax2.axvline(x=10, color='red', linestyle='--', alpha=0.5, label='10% outage')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=9)

if feasible.any():
    ax2.plot(best_out*100, best_area, 'r*', markersize=14, markeredgewidth=1.5)

plt.tight_layout()
plt.show()

# ==========================================
# 11. Report the knee solution details
# ==========================================
if feasible.any():
    feasible_indices = np.where(feasible)[0]
    best_idx = feasible_indices[np.argmin(res.F[feasible, 0])]
    best_x = res.X[best_idx]
    print("\n" + "="*60)
    print("Knee Solution (smallest area with outage <= 10%)")
    print("="*60)
    print(f"  xc={best_x[0]:.3f} m,  zc={best_x[1]:.3f} m")
    print(f"  Lx={best_x[2]:.3f} m,  Lz={best_x[3]:.3f} m")
    print(f"  Area = {res.F[best_idx,0]:.4f} m²")
    print(f"  Grid Outage = {res.F[best_idx,1]*100:.2f}%")
    print("="*60)

print("\nPareto front analysis complete.")